In [33]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Load color map ---
color_map = {}
with open("/Users/gloriaso/Desktop/BME499/NeuroCTA/NeuroCTA/Resources/Colors/labelmap_topbrain_ct.txt") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 6:
            continue
        try:
            r, g, b = int(parts[-4]), int(parts[-3]), int(parts[-2])
            name = " ".join(parts[1:-4])
            color_map[name] = f"rgb({r},{g},{b})"
        except ValueError:
            continue

# --- Load CSVs ---
vmtk_df = pd.read_csv("/Users/gloriaso/Desktop/BME499/vmtk_output/segment_features.csv")
skan_df = pd.read_csv("/Users/gloriaso/Desktop/BME499/skan_output/segment_features.csv")

# --- Rename VMTK columns ---
vmtk_df = vmtk_df.rename(columns={
    "length":      "length_vmtk",
    "tortuosity":  "tortuosity_vmtk",
    "mean_radius": "mean_radius_vmtk"
})

# --- Merge on segment name ---
df = vmtk_df.merge(skan_df, on="segment")
df["tortuosity_vmtk"] = df["tortuosity_vmtk"] + 1

# --- Clean inf and NaN ---
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["length_vmtk", "length", "tortuosity_vmtk",
                         "tortuosity_dm", "tortuosity_soam",
                         "mean_radius_vmtk", "mean_radius"])
print(f"Segments after cleaning: {len(df)}")
print(df["segment"].tolist())

df["color"] = df["segment"].map(color_map).fillna("rgb(150,150,150)")

# --- Plot ---
features = [
    ("length_vmtk",      "length",          "Length (mm)"),
    ("tortuosity_vmtk",  "tortuosity_dm",   "Tortuosity (DM)"),
    ("tortuosity_vmtk",  "tortuosity_soam", "Tortuosity (SOAM)"),
    ("mean_radius_vmtk", "mean_radius",     "Mean Radius (mm)"),
]

fig = make_subplots(rows=1, cols=4, subplot_titles=[f[2] for f in features])

for col_idx, (x_col, y_col, label) in enumerate(features, start=1):
    x = df[x_col]
    y = df[y_col]

    valid = df[[x_col, y_col]].dropna()
    r = np.corrcoef(valid[x_col], valid[y_col])[0, 1] if len(valid) > 1 else float("nan")

    # Scatter points colored by segment
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode="markers",
        text=df["segment"],
        textposition="top right",
        textfont=dict(size=8),
        marker=dict(
            color=df["color"],
            size=10,
            line=dict(color="black", width=0.5)
        ),
        name=label,
        showlegend=False,
        hovertemplate="%{text}<br>VMTK: %{x:.3f}<br>skan: %{y:.3f}<extra></extra>"
    ), row=1, col=col_idx)

    # Identity line
    lims = [
        min(valid[x_col].min(), valid[y_col].min()) * 0.95,
        max(valid[x_col].max(), valid[y_col].max()) * 1.05
    ]
    fig.add_trace(go.Scatter(
        x=lims, y=lims,
        mode="lines",
        line=dict(color="red", dash="dash"),
        showlegend=False
    ), row=1, col=col_idx)

    fig.update_xaxes(title_text="VMTK", title_standoff=5, range=lims, row=1, col=col_idx)
    fig.update_yaxes(title_text="skan", title_standoff=5, range=lims, row=1, col=col_idx)

    r_str = f"{r:.3f}" if not np.isnan(r) else "N/A"
    fig.layout.annotations[col_idx - 1].text = f"{label}"

    x_axis = "x" if col_idx == 1 else f"x{col_idx}"
    y_axis = "y" if col_idx == 1 else f"y{col_idx}"

    fig.add_annotation(
        x=0.98, y=0.05,
        xref=f"{x_axis} domain",
        yref=f"{y_axis} domain",
        text=f"r={r_str}",
        showarrow=False,
        font=dict(size=20, family="Times New Roman"),
        align="right",
        xanchor="right",
        bgcolor="white",
        bordercolor="black",
        borderwidth=1,
        borderpad=4,
    )

fig.update_layout(
    title="VMTK vs skan Feature Comparison",
    height=500,
    width=1400,
    font=dict(family="Times New Roman", size=15)
)

fig.show()

Segments after cleaning: 27
['BA', 'R-P1P2', 'L-P1P2', 'R-ICA', 'R-M1', 'L-ICA', 'L-M1', 'Acom', 'R-A1A2', 'L-A1A2', 'R-A3', 'L-A3', 'R-M3', 'L-M2', 'L-M3', 'L-P3P4', 'R-VA', 'L-VA', 'R-SCA', 'L-SCA', 'R-OA', 'VoG', 'StS', 'ICVs', 'R-BVR', 'L-BVR', 'SSS']


In [8]:
print(df[["length_vmtk", "length", "tortuosity_vmtk", "tortuosity_dm", "tortuosity_soam", "mean_radius_vmtk", "mean_radius"]].describe())
print(df[["length_vmtk", "length", "tortuosity_vmtk", "tortuosity_dm", "tortuosity_soam", "mean_radius_vmtk", "mean_radius"]].isnull().sum())

       length_vmtk      length  tortuosity_vmtk  tortuosity_dm  \
count    29.000000   28.000000        29.000000      28.000000   
mean     40.167690   71.929529              inf       1.370904   
std      66.424379   75.880746              NaN       0.248127   
min       0.257985    3.065421         1.000000       1.042086   
25%       9.944531   20.883006         1.032702       1.204502   
50%      28.001252   41.373687         1.070826       1.334834   
75%      34.726914   87.064932         1.368927       1.456807   
max     310.424897  278.420961              inf       2.090783   

       tortuosity_soam  mean_radius_vmtk  mean_radius  
count        28.000000         29.000000    28.000000  
mean          1.158071          1.966623     0.918910  
std           0.394010          5.323336     0.399673  
min           0.186778          0.178208     0.497002  
25%           0.993092          0.400761     0.576074  
50%           1.276337          0.884943     0.828314  
75%          

/Users/gloriaso/miniforge3/envs/syde575/lib/python3.13/site-packages/pandas/core/nanops.py:1016: RuntimeWarning:

invalid value encountered in subtract

